In [ ]:
import json
from urllib.parse import unquote
import pandas as pd
import numpy as np
import json
from sklearn.metrics import roc_auc_score, accuracy_score, matthews_corrcoef, classification_report, confusion_matrix
from imblearn.pipeline import Pipeline, make_pipeline
from imblearn.over_sampling import SMOTE
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.metrics import matthews_corrcoef, make_scorer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer

In [ ]:


def read_data():
    with open("./augmented_dataset.json") as json_file:
        dataset = json.load(json_file)

    comment_data = []
    for comment in dataset:
        comment_text = comment["comment"]
        label = comment["label"]
        code = comment["code"]
        comment_data.append([comment_text, code, label])

    return comment_data


def get_label_freq(comment_data):
    freq = {}
    for (comment, code, label) in comment_data:
        if label not in freq:
            freq[label] = 1
        else:
            freq[label] += 1

    return freq


# def remove_duplicates(comment_data):
#     unique_comment_data = []
#     seen = set()
#     for (comment, code, label) in comment_data:
#         if (comment, label) not in seen:
#             unique_comment_data.append([comment, code, label])
#             seen.add((comment, label))
#     return unique_comment_data

def filter_data(comment_data, freq_threshold):
    filtered_data = []
    freq = get_label_freq(comment_data)

    for (comment, code, label) in comment_data:
        if freq[label] >= freq_threshold:
            filtered_data.append([comment, code, label])

    return filtered_data

In [ ]:
data = read_data()
print(len(data))
get_label_freq(data)

2218


{'Task': 121,
 'Not a smell': 1257,
 'Beautification': 49,
 'Obvious': 675,
 'Commented out code': 35,
 'Vague': 52,
 'Misleading': 15,
 'Too much info': 6,
 'Nonlocal info': 4,
 'Irrelevant': 4}

remove duplicates

In [ ]:
# data = remove_duplicates(data)
# print(len(data))
# get_label_freq(data)

remove minority classes

In [ ]:
data = filter_data(data, 30)
print(len(data))
get_label_freq(data)

2189


{'Task': 121,
 'Not a smell': 1257,
 'Beautification': 49,
 'Obvious': 675,
 'Commented out code': 35,
 'Vague': 52}

In [ ]:
df = pd.DataFrame(data, columns = ['code', 'comment', 'label'])


In [ ]:
df.head()

,code,comment,label
0,TODO Tracked in https://github.com/ankidroid/A...,NA,Task
1,Only enable AnkiDroid notifications unrelated ...,public static final int PENDING_NOTIFICATI...,Not a smell
2,Other variables,NA,Beautification
3,----------------------------------------------...,NA,Beautification
4,Add a home button to the actionbar,getSupportActionBar().setHomeButtonEnabled(true);,Obvious


In [ ]:
df['text'] = df.apply(lambda row: row['code'] + ' ' + row['comment'] if row['comment'] != "NA" else row['code'], axis=1)


# Encode the labels
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['label'])

# Split the text and labels
X = df['text']
y = df['label']



In [ ]:
# Example: Print encoded and corresponding original labels
for encoded, original in zip(df['label'].unique(), label_encoder.inverse_transform(df['label'].unique())):
    print(f"Encoded: {encoded} -> Original: {original}")

Encoded: 4 -> Original: Task
Encoded: 2 -> Original: Not a smell
Encoded: 0 -> Original: Beautification
Encoded: 3 -> Original: Obvious
Encoded: 1 -> Original: Commented out code
Encoded: 5 -> Original: Vague


oversampling, 10 fold cross validation, mcc score, comment+code
apply vectorizer after train test splits

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
tfidf_vectorizer = TfidfVectorizer()


In [ ]:
# Define the models
models = {
    "Gradient Boosting": GradientBoostingClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Decision Tree": DecisionTreeClassifier(),
    "SVM": SVC(probability=True),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Naive Bayes": MultinomialNB(alpha=0.1),
    "KNN": KNeighborsClassifier(n_neighbors=3)
}

In [ ]:
# Define the Stratified K-Fold
stratified_kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=11)

# Dictionary to store cross-validation scores
cv_scores = {}

mcc scores

In [ ]:
# Train each model with 10-fold cross-validation
for model_name, model in models.items():
    print(f"\nTraining and evaluating {model_name}...")
    accuracy_scores = []
    mcc_scores = []
    reports = []
    matrices = []

    for fold, (train_index, val_index) in enumerate(stratified_kfold.split(X_train, y_train), start=1):
        print(f"  Fold {fold}...")
        X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
        y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

        # Apply TfidfVectorizer to the training fold
        vect = TfidfVectorizer()
        X_train_fold_tfidf = vect.fit_transform(X_train_fold)
        X_val_fold_tfidf = vect.transform(X_val_fold)
        print(f"    Vectorized training and validation data.")

        # Apply SMOTE to handle class imbalance
        smote = SMOTE(random_state=11)
        X_train_resampled, y_train_resampled = smote.fit_resample(X_train_fold_tfidf, y_train_fold)
        print(f"    Applied SMOTE to handle class imbalance.")

        # Train the model
        model.fit(X_train_fold_tfidf, y_train_fold)
        print(f"    Model trained on resampled data.")

        # Predict labels and calculate accuracy
        y_val_pred = model.predict(X_val_fold_tfidf)
        accuracy = accuracy_score(y_val_fold, y_val_pred)
        accuracy_scores.append(accuracy)
        mcc = matthews_corrcoef(y_val_fold, y_val_pred)
        mcc_scores.append(mcc)
        print(f"    Fold {fold} Accuracy: {accuracy}")
        print(f"    Fold {fold} MCC: {mcc}")

        # Generate and store classification report and confusion matrix
        report = classification_report(y_val_fold, y_val_pred)
        matrix = confusion_matrix(y_val_fold, y_val_pred)
        reports.append(report)
        matrices.append(matrix)

    mean_accuracy = np.mean(accuracy_scores)
    mean_mcc = np.mean(mcc_scores)
    cv_scores[model_name] = (mean_accuracy, mean_mcc)
    print(f'{model_name} - Cross-validation score: {mean_accuracy}')
    print(f'{model_name} - MCC score: {mean_mcc}')
    # Print classification reports and confusion matrices for the last fold
    print(f"\n{model_name} - Classification Report (Fold {fold}):\n{reports[-1]}")
    print(f"{model_name} - Confusion Matrix (Fold {fold}):\n{matrices[-1]}")

# Print all cross-validation scores
print("\nAll Cross-validation scores:")
for model_name, score in cv_scores.items():
    print(f'{model_name}: {score}')


Training and evaluating Gradient Boosting...
  Fold 1...
    Vectorized training and validation data.
    Applied SMOTE to handle class imbalance.
    Model trained on resampled data.
    Fold 1 Accuracy: 0.6420454545454546
    Fold 1 MCC: 0.3112741946724518
  Fold 2...
    Vectorized training and validation data.
    Applied SMOTE to handle class imbalance.
    Model trained on resampled data.
    Fold 2 Accuracy: 0.6514285714285715
    Fold 2 MCC: 0.35594109562479964
  Fold 3...
    Vectorized training and validation data.


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


    Applied SMOTE to handle class imbalance.
    Model trained on resampled data.
    Fold 3 Accuracy: 0.6571428571428571
    Fold 3 MCC: 0.35502888858386156
  Fold 4...
    Vectorized training and validation data.
    Applied SMOTE to handle class imbalance.
    Model trained on resampled data.
    Fold 4 Accuracy: 0.6914285714285714
    Fold 4 MCC: 0.4201883764238665
  Fold 5...
    Vectorized training and validation data.


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


    Applied SMOTE to handle class imbalance.
    Model trained on resampled data.
    Fold 5 Accuracy: 0.64
    Fold 5 MCC: 0.32341988912536784
  Fold 6...
    Vectorized training and validation data.


...

    Applied SMOTE to handle class imbalance.
    Model trained on resampled data.
    Fold 5 Accuracy: 0.6057142857142858
    Fold 5 MCC: 0.2557713942941536
  Fold 6...
    Vectorized training and validation data.


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


    Applied SMOTE to handle class imbalance.
    Model trained on resampled data.
    Fold 6 Accuracy: 0.6171428571428571
    Fold 6 MCC: 0.3085636902405626
  Fold 7...
    Vectorized training and validation data.


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


    Applied SMOTE to handle class imbalance.
    Model trained on resampled data.
    Fold 7 Accuracy: 0.5714285714285714
    Fold 7 MCC: 0.2598740202737986
  Fold 8...
    Vectorized training and validation data.
    Applied SMOTE to handle class imbalance.
    Model trained on resampled data.
    Fold 8 Accuracy: 0.56
    Fold 8 MCC: 0.20201283145302712
  Fold 9...
    Vectorized training and validation data.
    Applied SMOTE to handle class imbalance.
    Model trained on resampled data.
    Fold 9 Accuracy: 0.64
    Fold 9 MCC: 0.37092629007478295
  Fold 10...
    Vectorized training and validation data.
    Applied SMOTE to handle class imbalance.
    Model trained on resampled data.
    Fold 10 Accuracy: 0.6228571428571429
    Fold 10 MCC: 0.31020980273599225
KNN - Cross-validation score: 0.6082110389610389
KNN - MCC score: 0.29024578414753954

KNN - Classification Report (Fold 10):
              precision    recall  f1-score   support

           0       0.33      0.25      0.2

oversampling

In [ ]:
# Define MCC scorer
mcc_scorer = make_scorer(matthews_corrcoef)

# Perform 10-fold cross-validation
results = {}
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

for name, model in models.items():
    pipeline = Pipeline([
        ('vectorizer', TfidfVectorizer()),
        ('smote', SMOTE(random_state=42)),
        ('classifier', model)
    ])

    mcc_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring=mcc_scorer)
    accuracy_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='accuracy')

    results[name] = {
        "MCC": mcc_scores.mean(),
        "Accuracy": accuracy_scores.mean()
    }

# Display results
results_df = pd.DataFrame(results).T
print(results_df)